In [10]:
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from bs4 import BeautifulSoup

TEAM_CLASS = 'styled__TextStyled-sc-1mby3k1-0 jTXYPl'
SCORE_CLASS = 'styled__TextStyled-sc-1mby3k1-0 gqRVUt'

In [11]:
df = pd.read_csv('laliga_results_raw_2425.csv')
df['num_goals'] = df['home_score'] + df['away_score']

In [12]:
df = df.reset_index()[['gameday', 'home_team', 'away_team', 'home_score', 'away_score', 'num_goals']]

In [13]:
df

,gameday,home_team,away_team,home_score,away_score,num_goals
0,1,Athletic Club,Getafe CF,1,1,2
1,1,Real Betis,Girona FC,1,1,2
2,1,RC Celta,Deportivo Alavés,2,1,3
3,1,UD Las Palmas,Sevilla FC,2,2,4
4,1,CA Osasuna,CD Leganés,1,1,2
...,...,...,...,...,...,...
174,18,Valencia CF,Deportivo Alavés,2,2,4
175,18,Real Madrid,Sevilla FC,4,2,6
176,18,CD Leganés,Villarreal CF,2,5,7
177,18,UD Las Palmas,RCD Espanyol de Barcelona,1,0,1


In [14]:
# Round 1-10 for training, round 11 for testing
# Round 2-11 for training, round 12 for testing
# Round 3-12 for training, round 13 for testing

# Why discard? Because games played long ago does not have enough reference value
# Optimal script => can be used to generate the best testing strategy without any lookahead

# Hyperparameters: 
# lookback: number of days in the training set

In [16]:
roundFrom = 1
roundTo = 10
roundTest = roundTo + 1

In [18]:
teams = sorted(df.home_team.unique().tolist())
teams_dict = {}
for i in range(len(teams)):
    teams_dict[teams[i]] = i
df['home_team_id'] = df['home_team'].map(teams_dict)
df['away_team_id'] = df['away_team'].map(teams_dict)

In [19]:
df_train = df.query("gameday >= @roundFrom and gameday <= @roundTo")
df_val = df.query("gameday == @roundTest")

In [20]:
import torch
from torch.utils.data import Dataset, DataLoader
col_to_fit = 'num_goals'

class MatchDataset(Dataset):
    def __init__(self, df, col_to_fit):
        super().__init__()
        self.df = df[['home_team_id', 'away_team_id', col_to_fit]]
        self.x_home_away = list(zip(df.home_team_id.values, df.away_team_id.values))
        self.y_col = self.df[col_to_fit].values
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        return self.x_home_away[idx], self.y_col[idx]

BS = 64
ds_train = MatchDataset(df_train, col_to_fit)
ds_val = MatchDataset(df_val, col_to_fit)
dl_train = DataLoader(ds_train, BS, shuffle=True, num_workers=0)
dl_val = DataLoader(ds_val, BS, shuffle=True, num_workers=0)


In [21]:
from torch import nn
class MF(nn.Module):
    """ Matrix factorization model simple """
    def __init__(self, num_teams, emb_dim):
        super().__init__()
        self.home_emb = nn.Embedding(num_embeddings=num_teams, embedding_dim=emb_dim)
        self.home_emb.weight.data = torch.autograd.Variable(torch.rand((num_teams, emb_dim)))
        self.away_emb = nn.Embedding(num_embeddings=num_teams, embedding_dim=emb_dim)
        self.away_emb.weight.data = torch.autograd.Variable(torch.rand((num_teams, emb_dim)))
    def forward(self, home_id, away_id):
        home_emb = self.home_emb(home_id)
        away_emb = self.away_emb(away_id)
        element_product = (home_emb*away_emb).sum(1)
        return element_product

num_teams = 20
mdl = MF(num_teams, emb_dim=3)

In [22]:
import torch
from torch.utils.data import Dataset, DataLoader
col_to_fit = 'num_goals'

class MatchDataset(Dataset):
    def __init__(self, df, col_to_fit):
        super().__init__()
        self.df = df[['home_team_id', 'away_team_id', col_to_fit]]
        self.x_home_away = list(zip(df.home_team_id.values, df.away_team_id.values))
        self.y_col = self.df[col_to_fit].values
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        return self.x_home_away[idx], self.y_col[idx]

BS = 64
hs_ds_train = MatchDataset(df_train, 'home_score')
hs_ds_val = MatchDataset(df_val, 'home_score')
hs_dl_train = DataLoader(hs_ds_train, BS, shuffle=True, num_workers=0)
hs_dl_val = DataLoader(hs_ds_val, BS, shuffle=True, num_workers=0)

as_ds_train = MatchDataset(df_train, 'away_score')https://www.xiaohongshu.com/discovery/item/67713b3b000000000902c305?app_platform=android&ignoreEngage=true&app_version=8.67.0&share_from_user_hidden=true&xsec_source=app_share&type=video&xsec_token=CBbXT8-tu_lArtYTA1QBihDdcOoyLHvyzut-kVsud8cxM=&author_share=1&xhsshare=WeixinSession&shareRedId=N0w0RTw1NUs2NzUyOTgwNjY0OTc4PTg-&apptime=1735519739&share_id=b6eef6b87f9742778880376972c97c06
as_ds_val = MatchDataset(df_val, 'away_score')
as_dl_train = DataLoader(as_ds_train, BS, shuffle=True, num_workers=0)
as_dl_val = DataLoader(as_ds_val, BS, shuffle=True, num_workers=0)

ng_ds_train = MatchDataset(df_train, 'num_goals')
ng_ds_val = MatchDataset(df_val, 'num_goals')
ng_dl_train = DataLoader(ng_ds_train, BS, shuffle=True, num_workers=0)
ng_dl_val = DataLoader(ng_ds_val, BS, shuffle=True, num_workers=0)

In [24]:
def train(dl_train, dl_val, num_epoch):
    num_teams = 20
    mdl = MF(num_teams, emb_dim=2)
    LR = 0.0003
    NUM_EPOCHS = num_epoch
    device = 'cpu'
    opt = torch.optim.Adam(mdl.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    epoch_train_losses, epoch_val_losses = [], []

    for i in range(NUM_EPOCHS):
        train_losses, val_losses = [], []
        mdl.train()
        for xb,yb in dl_train:
            x_home = xb[0].to(device, dtype=torch.long)
            x_away = xb[1].to(device, dtype=torch.long)
            y = yb.to(device, dtype=torch.float)
            y_pred = mdl(x_home, x_away)
            loss = loss_fn(y_pred, y)
            train_losses.append(loss.item())
            opt.zero_grad()
            loss.backward()
            opt.step()
        mdl.eval()
        for xb,yb in dl_val:
            x_home = xb[0].to(device, dtype=torch.long)
            x_away = xb[1].to(device, dtype=torch.long)
            y = yb.to(device, dtype=torch.float)
            y_pred = mdl(x_home, x_away)
            loss = loss_fn(y_pred, y)
            val_losses.append(loss.item())
        # Start logging
        epoch_train_loss = np.mean(train_losses)
        epoch_val_loss = np.mean(val_losses)
        epoch_train_losses.append(epoch_train_loss)
        epoch_val_losses.append(epoch_val_loss)
        
#         print('Epoch: %s, train loss = %s, val loss = %s' % (i + 1, epoch_train_loss, epoch_val_loss))
    home_vec = mdl.home_emb.weight.data
    away_vec = mdl.away_emb.weight.data.t()
    complete_matrix = torch.matmul(home_vec, away_vec)
    complete_matrix = np.array(complete_matrix)
    complete_matrix = pd.DataFrame(data=complete_matrix, columns=teams, index=teams)
    return complete_matrix

In [25]:
complete_matrix = train(ng_dl_train, ng_dl_val, 100)

In [26]:
complete_matrix

,Athletic Club,Atlético de Madrid,CA Osasuna,CD Leganés,Deportivo Alavés,FC Barcelona,Getafe CF,Girona FC,RC Celta,RCD Espanyol de Barcelona,RCD Mallorca,Rayo Vallecano,Real Betis,Real Madrid,Real Sociedad,Real Valladolid CF,Sevilla FC,UD Las Palmas,Valencia CF,Villarreal CF
Athletic Club,1.791823,1.365544,1.299428,1.039276,1.257355,0.522643,1.032821,1.350598,0.867585,0.227295,0.539099,0.784627,1.536353,0.146846,0.736084,0.903731,1.599247,0.932545,0.307037,0.810167
Atlético de Madrid,0.335111,0.274616,0.281003,0.244306,0.193885,0.119996,0.203421,0.217701,0.221511,0.041111,0.071629,0.173540,0.279134,0.028419,0.117549,0.109891,0.313276,0.242211,0.064860,0.200206
CA Osasuna,1.213131,1.005614,1.039931,0.914223,0.677243,0.447679,0.742527,0.767264,0.837263,0.147990,0.241875,0.644226,1.005594,0.103451,0.413530,0.362517,1.142551,0.917304,0.239238,0.753829
CD Leganés,1.210458,0.880976,0.795830,0.594273,0.938491,0.305037,0.675568,0.987714,0.458179,0.156567,0.427210,0.472203,1.055575,0.097138,0.540681,0.738153,1.049751,0.483602,0.191363,0.442202
Deportivo Alavés,0.849784,0.665537,0.651656,0.539419,0.557853,0.268601,0.499384,0.608016,0.466674,0.106493,0.228466,0.397086,0.720986,0.070533,0.330348,0.373503,0.771668,0.505450,0.152545,0.429596
FC Barcelona,1.133502,0.929531,0.951771,0.828049,0.654411,0.406636,0.688412,0.735182,0.751261,0.139009,0.241294,0.587899,0.943883,0.096160,0.396925,0.369700,1.060125,0.821568,0.219638,0.678839
Getafe CF,0.797555,0.628349,0.618946,0.515919,0.515588,0.256394,0.470674,0.563901,0.449446,0.099677,0.208780,0.377860,0.675088,0.066383,0.306156,0.339116,0.726981,0.487492,0.144607,0.412605
Girona FC,1.607194,1.164429,1.046212,0.775300,1.257450,0.398888,0.894165,1.321051,0.592036,0.208267,0.575269,0.619594,1.403805,0.128712,0.723432,0.996367,1.389910,0.623438,0.252036,0.573732
RC Celta,1.580718,1.225823,1.188133,0.971790,1.063803,0.485554,0.922430,1.153078,0.830578,0.198977,0.443456,0.721675,1.346324,0.130597,0.627225,0.732188,1.426436,0.897295,0.279048,0.768296
RCD Espanyol de Barcelona,0.662447,0.453010,0.378013,0.249598,0.576108,0.133240,0.354179,0.593388,0.161011,0.087802,0.278013,0.217840,0.590101,0.051713,0.326362,0.493514,0.553021,0.161974,0.093464,0.168270


In [20]:
games = [('FC Barcelona', 'Real Valladolid CF'),
 ('Athletic Club', 'Atlético de Madrid'),
 ('RCD Espanyol de Barcelona', 'Rayo Vallecano'),
 ('Valencia CF', 'Villarreal CF'),
 ('CD Leganés', 'RCD Mallorca'),
 ('Deportivo Alavés', 'UD Las Palmas'),
 ('CA Osasuna', 'RC Celta'),
 ('Sevilla FC', 'Girona FC'),
 ('Getafe CF', 'Real Sociedad'),
 ('Real Madrid', 'Real Betis')]

In [23]:
# Simulation
from collections import defaultdict
from tqdm import tqdm

SIMU_TIMES = 100
res_count_dict = {}
for game in games:
    res_count_dict[game] = defaultdict(int)

for i in tqdm(range(SIMU_TIMES)):
    hs_matrix = train(hs_dl_train, hs_dl_val, 10)
    as_matrix = train(as_dl_train, as_dl_val, 10)
#     ng_matrix = train(ng_dl_train, ng_dl_val, 1000)
    for game in games:
        home_score = round(hs_matrix.loc[game])
        away_score = round(as_matrix.loc[game])
#         total_score = round(ng_matrix.loc[game])
        res_count_dict[game][(home_score, away_score)] += 1

  0%|                                                   | 0/100 [00:00<?, ?it/s]


IndexError: index out of range in self

In [41]:
for game in games:
    print(game)
    for res, count in res_count_dict[game].items():
        print("\t%s - %s appeared %s times" % (res[0], res[1], count))

('Athletic Club', 'Getafe CF')
	3 - 1 appeared 34 times
	2 - 1 appeared 64 times
	2 - 0 appeared 1 times
	3 - 0 appeared 1 times
('Real Betis', 'Girona FC')
	2 - 2 appeared 30 times
	1 - 2 appeared 54 times
	1 - 1 appeared 7 times
	2 - 1 appeared 9 times
('RC Celta', 'Deportivo Alavés')
	1 - 1 appeared 98 times
	1 - 0 appeared 1 times
	0 - 1 appeared 1 times
('UD Las Palmas', 'Sevilla FC')
	1 - 1 appeared 100 times
('Valencia CF', 'FC Barcelona')
	1 - 1 appeared 93 times
	1 - 2 appeared 7 times
('Real Sociedad', 'Rayo Vallecano')
	1 - 1 appeared 80 times
	1 - 0 appeared 15 times
	2 - 1 appeared 4 times
	2 - 0 appeared 1 times
('RCD Mallorca', 'Real Madrid')
	1 - 2 appeared 40 times
	1 - 1 appeared 41 times
	0 - 1 appeared 12 times
	0 - 2 appeared 7 times
('Villarreal CF', 'Atlético de Madrid')
	1 - 2 appeared 53 times
	2 - 2 appeared 37 times
	1 - 3 appeared 7 times
	2 - 3 appeared 3 times
